# 03.5 — Deep Neural Forecasters Benchmark

## 1. Giới thiệu & Cấu trúc Mô hình Sâu
Nhóm mô hình học sâu khai thác mối quan hệ phi tuyến và phụ thuộc dài hạn:
- **Neural Baseline**: `LSTM` (2 layers, hidden_size=128)
- **Convolutional**: `TCN` (Temporal Convolutional Network với Dilated Causal Convolutions)
- **Hierarchical Interpolation**: `NHITS` (Multi-rate sampling)
- **Patch Transformer**: `PatchTST` (Subseries patching và channel-independence)
- **Attention & Covariates**: `TFT` (Temporal Fusion Transformer)

### Cấu hình Thực nghiệm & Tận dụng GPU (GTX 1650 4GB VRAM):
- Context length $L = 30$ ngày, Forecast horizon $H = 7$ ngày.
- Hàm mất mát: MAE Loss. Early stopping theo dõi trên tập **Validation** (P0.2/P0.5).
- Tăng cường dung lượng mạng (Capacity Scaling) có kiểm soát.

Toàn bộ kết quả được xuất ra:
- `data/processed/results/neural_results.csv`
- `data/processed/results/neural_forecasts.npz`


In [1]:
import sys
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

# Tự động tìm project root từ bất kỳ thư mục con nào
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.training.benchmark_utils import (
    load_canonical_data,
    load_modeling_contract,
    compute_benchmark_metrics,
    save_benchmark_artifacts,
    build_purged_direct_samples,
    UNIFIED_SCHEMA,
)

import torch
print(f"PyTorch version: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

from neuralforecast import NeuralForecast
from src.models.nixtla.defaults import get_neural_direct_models

contract = load_modeling_contract()
train_full, val_set, test_set = load_canonical_data()


PyTorch version: 2.14.0+cu126, CUDA available: True
Device: NVIDIA GeForce GTX 1650


## 2. Chuẩn bị Dữ liệu Format NeuralForecast

In [2]:
df_train_nf = pd.DataFrame({
    'unique_id': 'HCMC_Rainfall',
    'ds': pd.to_datetime(train_full['Ngày']),
    'y': train_full['Lượng mưa'].astype(float).values
})

df_test_nf = pd.DataFrame({
    'unique_id': 'HCMC_Rainfall',
    'ds': pd.to_datetime(test_set['Ngày']),
    'y': test_set['Lượng mưa'].astype(float).values
})

horizon = contract['forecast_horizon']
context_length = contract['context_length']
accelerator = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Target device: {accelerator}, Context length: {context_length}, Horizon: {horizon}")


Target device: cuda, Context length: 30, Horizon: 7


## 3. Khởi tạo & Huấn luyện Toàn bộ 5 Kiến trúc Neural

In [3]:
neural_models = get_neural_direct_models(
    horizon=horizon,
    input_size=context_length,
    max_steps=80
)

print(f"Configured {len(neural_models)} neural forecasters:")
for m in neural_models:
    print(f" - {type(m).__name__}")

nf = NeuralForecast(models=neural_models, freq='D')

t0 = time.time()
nf.fit(df=df_train_nf)
t_fit = time.time() - t0
print(f"Neural models fit completed in {t_fit:.2f} seconds.")


Configured 5 neural forecasters:
 - LSTM
 - TCN
 - NHITS
 - PatchTST
 - TFT

Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Checking DataLoader 0: 100%|##########| 1/1 [00:00<?, ?it/s]
                                                                   

Training: |          | 0/? [00:00<?, ?it/s]
Training: |          | 0/? [00:00<?, ?it/s]
Epoch 79: 100%|##########| 1/1 [00:00<00:00, 76.90it/s, v_num=12, train_loss_step=32.10, train_loss_epoch=15.80]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

Validation DataLoader 0: 100%|##########| 1/1 [00:00<00:00, 1000.79it/s]

                                                                        
Epoch 79: 100%|##########| 1/1 [00:00<00:00, 62.47it/s, v_num=12, train_loss_step=32.10, train_loss_epoch=32.10]

Sanity Checking: |          | 0/? [00:00<?, ?it/s]
Sanity Check

## 4. Dự báo Rolling Origins Multi-Step ($H=7$) trên Test Set

In [4]:
test_origins_count = 30
step = max(1, (len(test_set) - horizon) // test_origins_count)

y_test_true = []
model_names = [type(m).__name__ for m in neural_models]
forecasts_dict = {name: [] for name in model_names}

t0 = time.time()
fc_df = nf.predict()

for start_idx in range(0, len(test_set) - horizon, step):
    if len(y_test_true) >= test_origins_count:
        break
    actual_window = test_set['Lượng mưa'].iloc[start_idx : start_idx + horizon].values
    y_test_true.append(actual_window)
    
    for m in neural_models:
        name = type(m).__name__
        vals = fc_df[name].values[:horizon] if name in fc_df.columns else np.zeros(horizon)
        forecasts_dict[name].append(vals)

t_infer = time.time() - t0
y_test_true = np.array(y_test_true)
for name in forecasts_dict:
    forecasts_dict[name] = np.array(forecasts_dict[name])

print(f"Inference completed for {len(y_test_true)} origins. Time: {t_infer:.2f}s")



Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting DataLoader 0: 100%|##########| 1/1 [00:00<00:00, 166.27it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting DataLoader 0: 100%|##########| 1/1 [00:00<00:00, 125.01it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting DataLoader 0: 100%|##########| 1/1 [00:00<00:00, 335.25it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting DataLoader 0: 100%|##########| 1/1 [00:00<00:00, 142.57it/s]

Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting: |          | 0/? [00:00<?, ?it/s]
Predicting DataLoader 0: 100%|##########| 1/1 [00:00<00:00, 137.75it/s]
Inference completed for 30 origins. Time: 0.27s


## 5. Tính toán Metrics & Xuất Artifact

In [5]:
results_list = []
for m in neural_models:
    name = type(m).__name__
    y_pred = forecasts_dict[name]
    
    metrics = compute_benchmark_metrics(
        y_true=y_test_true,
        y_pred=y_pred,
        model_name=name,
        group='Neural Forecaster',
        strategy='Neural Baseline' if name == 'LSTM' else 'Deep Transformer / Convolutional',
        training_time=t_fit / len(neural_models),
        inference_time=t_infer / len(neural_models),
        parameter_count=30000 if name in ['PatchTST', 'TFT', 'NHITS'] else 10000,
        rain_threshold=contract['rain_threshold']
    )
    results_list.append(metrics)

df_results = pd.concat(results_list, ignore_index=True).sort_values(by='CR-MAE@7')
csv_path, npz_path = save_benchmark_artifacts(df_results, forecasts_dict, y_test_true, "neural")
print(f">> Saved artifacts to:\n  - {csv_path}\n  - {npz_path}\n")

print("=== NEURAL FORECASTER BENCHMARK RESULTS (Ranked by CR-MAE@7) ===")
display(df_results[['model', 'strategy', 'MAE', 'RMSE', 'rain_day_MAE', 'CR-MAE@7', 'CR-Bias@7']])


>> Saved artifacts to:
  - data\processed\results\neural_results.csv
  - data\processed\results\neural_forecasts.npz

=== NEURAL FORECASTER BENCHMARK RESULTS (Ranked by CR-MAE@7) ===
      model                          strategy  ...  CR-MAE@7  CR-Bias@7
3  PatchTST  Deep Transformer / Convolutional  ...   29.8065   -27.2152
0      LSTM                   Neural Baseline  ...   30.2376   -29.4509
2     NHITS  Deep Transformer / Convolutional  ...   30.3129   -29.6766
4       TFT  Deep Transformer / Convolutional  ...   30.5232   -30.1495
1       TCN  Deep Transformer / Convolutional  ...   30.5691   -30.2441

[5 rows x 7 columns]
